In [1]:
from langgraph.graph import StateGraph, START, END 
from typing import TypedDict, NotRequired

from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv



In [2]:
load_dotenv()

True

In [3]:
model = ChatGoogleGenerativeAI(
    model = "gemini-3.5-flash-lite"
)

In [4]:
# Defining a state 

class BlogState(TypedDict):
    title: str 
    outline: NotRequired[str] 
    content: NotRequired[str] 
    evaluate: NotRequired[str]



In [5]:
def create_outline(state: BlogState) -> BlogState:
    # fetch title 

    title = state['title']

    # create prompt for getting the outline -> give the prompt to the llm 

    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content[0]['text'] # type: ignore

    #  update state

    state['outline'] = outline 

    return state



In [ ]:
def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline'] # type: ignore

    prompt = f'write a detailed blog on the title - {title} using the following outline \n {outline}'

    content = model.invoke(prompt).content[0]['text'] # type: ignore

    state['content'] = content 

    return state




In [7]:
def evaluate_blog(state: BlogState) -> BlogState:
    """
    Based on the outline, this function will evaluate the blog created. It will output a integer score out of 10.
    """

    outline = state['outline'] # type: ignore 
    content = state['content'] # type: ignore

    prompt = f'Rate the blog with a score between 1 and 10 based on this outline - \n{outline} \n\n blog -\n{content}.'

    evaluation = model.invoke(prompt).content[0]['text'] # type: ignore

    state['evaluate'] = evaluation 

    return state 


        

In [8]:
graph = StateGraph(BlogState)

# add node 
graph.add_node('create_outline',create_outline)
graph.add_node('create_blog',create_blog)
graph.add_node('evaluate_blog', evaluate_blog)

# edges 

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline','create_blog')
graph.add_edge('create_blog', 'evaluate_blog')
graph.add_edge('evaluate_blog',END)

workflow = graph.compile()

In [10]:
initial_state: BlogState = {'title': "Rise of AI in India"}

final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'Rise of AI in India', 'outline': 'Here is a comprehensive and detailed blog post outline on the **"Rise of AI in India."** \n\nThis outline is structured for a 1,500 to 2,000-word in-depth article, balancing economic impact, technological advancements, social implications, and future outlook.\n\n---\n\n# Blog Title Options:\n*   *The AI Revolution in India: How the Subcontinent is Shaping the Future of Intelligence*\n*   *From Silicon Valley to Silicon Ganges: The Meteoric Rise of AI in India*\n*   *Bharat 2.0: How Artificial Intelligence is Transforming India’s Economy and Society*\n\n---\n\n## I. Introduction (The Hook)\n*   **The Hook:** Start with a relatable scenario—interacting with an AI chatbot in a regional Indian language, or using UPI seamlessly backed by AI fraud detection.\n*   **The Big Picture:** India is no longer just an IT back-office; it is rapidly transforming into an AI innovation powerhouse.\n*   **Thesis Statement:** Driven by a massive talent pool, go

In [11]:
print(final_state['outline'])

Here is a comprehensive and detailed blog post outline on the **"Rise of AI in India."** 

This outline is structured for a 1,500 to 2,000-word in-depth article, balancing economic impact, technological advancements, social implications, and future outlook.

---

# Blog Title Options:
*   *The AI Revolution in India: How the Subcontinent is Shaping the Future of Intelligence*
*   *From Silicon Valley to Silicon Ganges: The Meteoric Rise of AI in India*
*   *Bharat 2.0: How Artificial Intelligence is Transforming India’s Economy and Society*

---

## I. Introduction (The Hook)
*   **The Hook:** Start with a relatable scenario—interacting with an AI chatbot in a regional Indian language, or using UPI seamlessly backed by AI fraud detection.
*   **The Big Picture:** India is no longer just an IT back-office; it is rapidly transforming into an AI innovation powerhouse.
*   **Thesis Statement:** Driven by a massive talent pool, government backing, and unique societal challenges, India’s AI 

In [12]:
print(final_state['content'])

# Bharat 2.0: How Artificial Intelligence is Transforming India’s Economy and Society

Imagine waking up in a remote village in rural India. You pick up your feature-loaded smartphone, speak into it in your native regional dialect, and instantly receive real-time advice on treating a fungal infection destroying your mustard crop. Minutes later, you transfer money to a local vendor using a UPI app equipped with real-time, AI-driven fraud detection. 

This isn't a scene from a sci-fi novel. This is the everyday reality of modern India. 

For decades, the global narrative pigeonholed India as the world’s back-office—a vast reservoir of coding talent executing outsourced Western IT tasks. Today, that script has been aggressively rewritten. Driven by a massive demographic dividend, visionary public policy, and a unique set of socio-economic challenges, India is emerging as a global artificial intelligence (AI) powerhouse. 

The subcontinent's AI boom is not merely a technological upgrade; i

In [13]:
type(final_state)

dict

In [14]:
final_state.keys()

dict_keys(['title', 'outline', 'content', 'evaluate'])

In [15]:
print(final_state['evaluate'])

Based on the comprehensive outline provided and the quality of the written blog post, here is my rating:

### **Score: 9.5 / 10**

---

### **Detailed Evaluation:**

*   **Adherence to Outline (10/10):** The blog follows the provided outline meticulously. Every major section (Drivers, Sectors, Ecosystem, Challenges, Future Outlook) is addressed thoroughly, incorporating specific examples mentioned in the notes (like the Bhashini project, UPI, Sarvam AI, and Krutrim).
*   **Tone & Style (10/10):** The tone strikes the exact right balance requested in the tips—it is optimistic yet pragmatic, analytical, and forward-looking. The writing avoids blind hype by acknowledging genuine hurdles (infrastructure, digital divide, job displacement) in Section V.
*   **Hook & Narrative Flow (10/10):** The opening hook in Section I brilliantly brings the abstract concept of "AI in India" down to a relatable, vivid human experience. The transition between the macroeconomic drivers and grassroots sector 